In [ ]:
import cv2
import numpy as np
import os
import time
import random
import urllib.request
from tensorflow.keras.models import load_model
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model = load_model("emotion_model.keras")
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

emotion_colors = {
    'angry': (0, 0, 255),
    'disgust': (0, 150, 0),
    'fear': (200, 0, 200),
    'happy': (0, 220, 255),
    'neutral': (200, 200, 200),
    'sad': (255, 100, 0),
    'surprise': (0, 255, 255)
}

face_model_path = "blaze_face_short_range.tflite"
if not os.path.exists(face_model_path):
    url = "https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite"
    urllib.request.urlretrieve(url, face_model_path)

base_options = python.BaseOptions(model_asset_path=face_model_path)
options = vision.FaceDetectorOptions(base_options=base_options)
detector = vision.FaceDetector.create_from_options(options)


def draw_bar_chart(panel, probs):
    panel[:] = (30, 30, 30)
    bar_height = 40
    max_width = panel.shape[1] - 120
    for i, emotion in enumerate(emotion_labels):
        y = 20 + i * (bar_height + 15)
        prob = probs[i]
        color = emotion_colors[emotion]
        cv2.putText(panel, emotion, (10, y + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
        bar_w = int(max_width * prob)
        cv2.rectangle(panel, (110, y), (110 + bar_w, y + bar_height), color, -1)
        cv2.putText(panel, f"{prob*100:.0f}%", (115 + bar_w, y + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)


def apply_emotion_effect(frame, emotion, t):
    h, w = frame.shape[:2]
    if emotion == "happy":
        for _ in range(15):
            x = random.randint(0, w - 1)
            y = random.randint(0, h - 1)
            size = random.randint(3, 7)
            color = random.choice([(0, 255, 255), (255, 0, 255), (0, 255, 0), (255, 255, 0)])
            cv2.circle(frame, (x, y), size, color, -1)
    elif emotion == "angry":
        thickness = 8 + int(6 * abs(np.sin(t * 6)))
        cv2.rectangle(frame, (0, 0), (w - 1, h - 1), (0, 0, 255), thickness)
    elif emotion == "sad":
        overlay = frame.copy()
        overlay[:] = (150, 50, 0)
        frame[:] = cv2.addWeighted(overlay, 0.15, frame, 0.85, 0)
    elif emotion == "surprise":
        radius = int(50 + 100 * abs(np.sin(t * 4)))
        cv2.circle(frame, (w // 2, h // 2), radius, (0, 255, 255), 3)
    elif emotion == "fear":
        overlay = frame.copy()
        overlay[:] = (150, 0, 150)
        frame[:] = cv2.addWeighted(overlay, 0.15, frame, 0.85, 0)
    elif emotion == "disgust":
        overlay = frame.copy()
        overlay[:] = (0, 100, 0)
        frame[:] = cv2.addWeighted(overlay, 0.15, frame, 0.85, 0)


cap = cv2.VideoCapture(0)
ret, sample = cap.read()
fh, fw = sample.shape[:2] if ret else (480, 640)
panel_width = 320

challenge_mode = False
challenge_emotion = None
challenge_start_time = 0
challenge_duration = 6
score = 0
message = ""
message_timer = 0

last_probs = np.array([1/7] * 7)
frame_count = 0

print("Controls: c = start emotion challenge | q = quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    t = time.time()

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = detector.detect(mp_image)

    dominant_emotion = "neutral"

    if result.detections:
        detection = result.detections[0]
        bbox = detection.bounding_box
        x, y = max(0, bbox.origin_x), max(0, bbox.origin_y)
        bw, bh = bbox.width, bbox.height
        face_roi = frame[y:y + bh, x:x + bw]
        if face_roi.size != 0 and frame_count % 3 == 0:
            gray = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, (48, 48))
            gray = gray.astype("float32") / 255.0
            gray = np.expand_dims(gray, axis=(0, -1))
            prediction = model.predict(gray, verbose=0)
            last_probs = prediction[0]

        dominant_idx = np.argmax(last_probs)
        dominant_emotion = emotion_labels[dominant_idx]
        confidence = last_probs[dominant_idx] * 100

        cv2.rectangle(frame, (x, y), (x + bw, y + bh), emotion_colors[dominant_emotion], 2)
        cv2.putText(frame, f"{dominant_emotion} ({confidence:.0f}%)", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, emotion_colors[dominant_emotion], 2)

        apply_emotion_effect(frame, dominant_emotion, t)

    if challenge_mode:
        remaining = challenge_duration - (t - challenge_start_time)
        if dominant_emotion == challenge_emotion:
            score += 10
            message = f"Success! +10 points (Total: {score})"
            message_timer = t + 2
            challenge_mode = False
        elif remaining <= 0:
            message = "Time's up! Try again."
            message_timer = t + 2
            challenge_mode = False
        else:
            cv2.putText(frame, f"Show: {challenge_emotion.upper()}  ({remaining:.1f}s)", (20, fh - 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    if t < message_timer:
        cv2.putText(frame, message, (20, fh - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    cv2.putText(frame, f"Score: {score}", (fw - 160, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    canvas = np.zeros((fh, fw + panel_width, 3), dtype=np.uint8)
    canvas[:, :fw] = frame
    draw_bar_chart(canvas[:, fw:], last_probs)

    cv2.imshow("Emotion Detector", canvas)

    frame_count += 1
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    elif key == ord('c') and not challenge_mode:
        challenge_emotion = random.choice(emotion_labels)
        challenge_start_time = time.time()
        challenge_mode = True

cap.release()
cv2.destroyAllWindows()